# S2 α-sweep + SSCV + raw-score archival

Extends `multiscale_ablation.ipynb` (cell 18 pipeline) with:

1. **α sweep**: D/A/B/C at α ∈ {0.05, 0.10, 0.15}. CV for (r, bw) done at α=0.10.
2. **SSCV**: per-(tile, size, α, method) size-stratified coverage violation.
3. **Archive raw scores**: softmax probs, APS scores, SACP-smoothed scores per r, per-pixel `in_set_{D,A,B,C}` at each α. Any future re-analysis skips XGBoost retraining.

Runs 10 tiles × 3 sizes = 30 configs (1 degenerate, 29 usable). XGBoost is CPU-fast; ~30–60 min total on a Colab CPU runtime.

Outputs: `/content/s2_alpha_sweep/checkpoints/{tile}_s{size}.pkl` mirrored to `/content/drive/MyDrive/s2_alpha_sweep/checkpoints/`.


## 1. Install dependencies

In [1]:
!pip install -q xgboost scikit-learn pandas matplotlib scipy

## 2. Mount Drive + paths (local primary, Drive mirror)

In [2]:
import os, sys, time, pickle, json, shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# --- source tiles (from earlier S2 pipeline, on Drive) ---
WORK_DIR   = '/content/drive/MyDrive/sentinel2_landcover_pilot_10m'
TILE_DIR   = f'{WORK_DIR}/tiles'

# --- LOCAL primary workspace ---
LOCAL_DIR   = '/content/s2_alpha_sweep'
CKPT_DIR    = f'{LOCAL_DIR}/checkpoints'
RESULTS_DIR = f'{LOCAL_DIR}/results'
for d in [LOCAL_DIR, CKPT_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)

# --- DRIVE mirror ---
DRIVE_MIRROR   = '/content/drive/MyDrive/s2_alpha_sweep'
DRIVE_CKPT_DIR = f'{DRIVE_MIRROR}/checkpoints'
try: os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
except OSError: pass

# Restore cached pkls from Drive to local on cold start
def _restore():
    if not os.path.isdir(DRIVE_CKPT_DIR): return 0
    try: pkls = [f for f in os.listdir(DRIVE_CKPT_DIR) if f.endswith('.pkl')]
    except OSError: return 0
    n = 0
    for f in pkls:
        src, dst = f'{DRIVE_CKPT_DIR}/{f}', f'{CKPT_DIR}/{f}'
        if os.path.exists(dst): continue
        try: shutil.copy2(src, dst); n += 1
        except OSError as e: print(f'[restore] {f}: {e}')
    return n
print(f'[restore] copied {_restore()} cached pkls from Drive to local')

# Sanity
if not os.path.isdir(TILE_DIR):
    raise RuntimeError(f'TILE_DIR missing: {TILE_DIR}')
tiles_present = sorted(f.replace('.npz','') for f in os.listdir(TILE_DIR) if f.endswith('.npz'))
print(f'TILE_DIR : {TILE_DIR}  ({len(tiles_present)} tiles)')
print(f'CKPT_DIR : {CKPT_DIR}  (local, primary)')
print(f'MIRROR   : {DRIVE_CKPT_DIR}  (Drive, best-effort backup)')


Mounted at /content/drive
[restore] copied 0 cached pkls from Drive to local
TILE_DIR : /content/drive/MyDrive/sentinel2_landcover_pilot_10m/tiles  (10 tiles)
CKPT_DIR : /content/s2_alpha_sweep/checkpoints  (local, primary)
MIRROR   : /content/drive/MyDrive/s2_alpha_sweep/checkpoints  (Drive, best-effort backup)


## 3. Tile config + grid

In [3]:
TILES = {
    'polk_iowa':      'Polk County, Iowa (row-crop)',
    'lancaster_pa':   'Lancaster, Pennsylvania (mixed ag)',
    'hartford_ct':    'Hartford, Connecticut (urban-forest)',
    'everglades_fl':  'Everglades, Florida (wetland)',
    'lubbock_tx':     'Lubbock, Texas (irrigated dryland)',
    'sacramento_ca':  'Sacramento Delta, California',
    'phoenix_az':     'Phoenix, Arizona (desert-urban)',
    'yellowstone_wy': 'Yellowstone, Wyoming (forest)',
    'seattle_wa':     'Seattle, Washington (urban-water)',
    'mississippi_la': 'Mississippi Delta, Louisiana',
}
SIZES        = [100, 200, 500]          # 1, 2, 5 km at 10 m/px
RADIUS_GRID  = [1, 2, 3, 5, 10]
BW_GRID      = [3, 5, 10, 20, 50]
ALPHA_GRID   = (0.05, 0.10, 0.15)
ALPHA_CV     = 0.10
MAX_CAL      = 20000
CV_FOLDS     = 5
LMD          = 0.5
SEED         = 0
print(f'{len(TILES)} tiles x {len(SIZES)} sizes = {len(TILES)*len(SIZES)} configs; α grid: {ALPHA_GRID}')


10 tiles x 3 sizes = 30 configs; α grid: (0.05, 0.1, 0.15)


## 4. Helpers (adapted from `multiscale_ablation.ipynb` cells 8 + 18)

In [4]:
import numpy as np
import pandas as pd
import xgboost as xgb
from collections import Counter
from sklearn.model_selection import train_test_split, KFold
from scipy.spatial.distance import cdist
from scipy.signal import convolve2d
from scipy import stats as sstats

def aps_scores(probs, labels=None, rng=None):
    rng = rng or np.random
    n, K = probs.shape
    si = np.argsort(-probs, axis=1)
    sp = np.take_along_axis(probs, si, axis=1)
    cs = np.cumsum(sp, axis=1)
    U = rng.uniform(0, 1, size=(n, K))
    ss = cs - sp * U
    scores = np.zeros_like(ss)
    for i in range(n):
        scores[i, si[i]] = ss[i]
    if labels is not None:
        return np.array([scores[i, int(labels[i])] for i in range(n)])
    return scores

def conformal_quantile(scores, alpha):
    n = len(scores)
    return float(np.quantile(scores, np.ceil((n+1)*(1-alpha))/n))

def set_interval_score_vec(score_mat, q_vec, y_true, alpha):
    in_set = score_mat < q_vec[:, None]
    sizes  = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    penalty = (2.0 / alpha) * (~covered).astype(np.float64)
    return float((sizes + penalty).mean())

def eval_method(score_mat, q_scalar_or_vec, y_true, alpha):
    q_vec = np.full(len(y_true), q_scalar_or_vec) if np.isscalar(q_scalar_or_vec) else q_scalar_or_vec
    in_set = score_mat < q_vec[:, None]
    sizes  = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    size_mean = float(sizes.mean())
    cov_mean  = float(covered.mean())
    IS = float((sizes + (2.0/alpha)*(~covered).astype(np.float64)).mean())
    # SSCV: bucket by size, take max |cov - (1-alpha)| across nonempty buckets
    buckets = [(1,1),(2,2),(3,3),(4,4),(5,10**9)]
    worst = 0.0; rows = []
    for lo, hi in buckets:
        m = (sizes >= lo) & (sizes <= hi)
        if m.sum() == 0: rows.append((lo, hi, 0, None)); continue
        c = float(covered[m].mean())
        rows.append((lo, hi, int(m.sum()), c))
        worst = max(worst, abs(c - (1 - alpha)))
    return dict(is_=IS, cov=cov_mean, size=size_mean, sscv_pct=worst*100,
                sscv_per_bucket=rows, in_set=in_set, sizes=sizes, covered=covered)

def sacp_smooth_radius(score_map, H, W, valid_idx, radius, lmd=0.5, k_iter=1):
    N, K = score_map.shape
    if radius <= 0: return score_map.copy()
    mask = np.zeros(N, dtype=bool); mask[valid_idx] = True
    mask_2d = mask.reshape(H, W).astype(np.float64)
    score_2d = score_map.reshape(H, W, K).astype(np.float64)
    k_size = 2 * radius + 1
    kernel = np.ones((k_size, k_size), dtype=np.float64); kernel[radius, radius] = 0.0
    for _ in range(k_iter):
        den = convolve2d(mask_2d, kernel, mode='same', boundary='fill', fillvalue=0.0)
        smoothed = np.empty_like(score_2d)
        for cls in range(K):
            values = np.where(mask_2d > 0, score_2d[..., cls], 0.0)
            num = convolve2d(values, kernel, mode='same', boundary='fill', fillvalue=0.0)
            smoothed[..., cls] = np.where(den > 1e-10, num / np.maximum(den, 1e-10), 0.0)
        new_score = (1 - lmd) * score_2d + lmd * smoothed
        score_2d = np.where(mask_2d[..., None] > 0, new_score, score_2d)
    return score_2d.reshape(N, K)

def vwq(sorted_scores, d_matrix, order, bw, alpha):
    log_w = -0.5 * (d_matrix / bw) ** 2
    log_w -= log_w.max(axis=1, keepdims=True)
    w = np.exp(log_w); w_sorted = w[:, order]
    ws = w_sorted / w_sorted.sum(axis=1, keepdims=True)
    cum = np.cumsum(ws, axis=1)
    k_star = np.argmax(cum >= (1 - alpha), axis=1)
    return sorted_scores[k_star]

def stratified_sub(labels, n, seed):
    if n is None or n >= len(labels): return np.arange(len(labels))
    uniq, cnts = np.unique(labels, return_counts=True)
    per_cls_n = np.maximum(1, (n * cnts / cnts.sum()).astype(int))
    rng = np.random.RandomState(seed)
    out = []
    for c, nc in zip(uniq, per_cls_n):
        pool = np.where(labels == c)[0]
        out.append(pool if len(pool) <= nc else rng.choice(pool, size=nc, replace=False))
    return np.concatenate(out)

print('Helpers loaded.')


Helpers loaded.


## 5. α-sweep experiment function (per tile × size)

In [5]:
def run_tile_size(tile_key, size_px, alpha_grid=ALPHA_GRID, alpha_cv=ALPHA_CV,
                   radius_grid=RADIUS_GRID, bw_grid=BW_GRID,
                   max_cal=MAX_CAL, cv_folds=CV_FOLDS, lmd=LMD, seed=SEED):
    ckpt_path = f'{CKPT_DIR}/{tile_key}_s{size_px}.pkl'
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'rb') as f: return pickle.load(f)

    npz_path = f'{TILE_DIR}/{tile_key}.npz'
    arr = np.load(npz_path)
    emb_full, label_full = arr['emb'], arr['label']
    H_full, W_full = label_full.shape
    Seff = min(size_px, H_full, W_full)
    r0 = H_full // 2 - Seff // 2
    c0 = W_full // 2 - Seff // 2
    emb   = emb_full[r0:r0+Seff, c0:c0+Seff]
    label = label_full[r0:r0+Seff, c0:c0+Seff]

    H, W, D = emb.shape; N = H * W
    flat_label = label.ravel()
    X_flat = emb.reshape(N, D)
    labeled_idx = np.where(flat_label > 0)[0]
    y_raw = flat_label[labeled_idx]
    counts = Counter(y_raw.tolist())
    min_count = int(max(10, min(100, N // 500)))
    rare = [c for c, cnt in counts.items() if cnt < min_count]
    keep = ~np.isin(y_raw, rare)
    labeled_idx = labeled_idx[keep]; y_raw = y_raw[keep]
    X_lab = X_flat[labeled_idx]
    classes = sorted(np.unique(y_raw).tolist()); K = len(classes)
    if K < 2 or len(y_raw) < 200:
        return {'tile': tile_key, 'size_px': Seff, 'degenerate': True, 'n_classes': K}
    cls_remap = {c: i for i, c in enumerate(classes)}
    y = np.array([cls_remap[v] for v in y_raw])

    idx_pos = np.arange(len(y))
    idx_tr, idx_tmp = train_test_split(idx_pos, train_size=0.6, random_state=seed*100+42, stratify=y)
    idx_ca, idx_te  = train_test_split(idx_tmp, test_size=0.5, random_state=seed*100+42, stratify=y[idx_tmp])

    model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              objective='multi:softprob', num_class=K,
                              tree_method='hist', random_state=seed*100+42, verbosity=0)
    model.fit(X_lab[idx_tr], y[idx_tr])
    probs_ca = model.predict_proba(X_lab[idx_ca])
    probs_te = model.predict_proba(X_lab[idx_te])
    pred_te  = np.argmax(probs_te, axis=1)
    acc = float((pred_te == y[idx_te]).mean())

    rng = np.random.RandomState(seed*100+42)
    cal_all  = aps_scores(probs_ca, rng=rng)
    test_all = aps_scores(probs_te, rng=rng)
    cal_true = np.array([cal_all[i, int(y[idx_ca[i]])] for i in range(len(idx_ca))])

    cal_flat_idx  = labeled_idx[idx_ca]
    test_flat_idx = labeled_idx[idx_te]

    # SACP-smoothed full score maps per r
    score_map = np.zeros((N, K), dtype=np.float64)
    score_map[cal_flat_idx]  = cal_all
    score_map[test_flat_idx] = test_all
    valid_idx_all = np.concatenate([cal_flat_idx, test_flat_idx])

    fused_per_r = {r: sacp_smooth_radius(score_map, H, W, valid_idx_all, radius=r, lmd=lmd)
                   for r in radius_grid}
    fcu_per_r = {r: np.array([fused_per_r[r][cal_flat_idx[e], int(y[idx_ca[e]])]
                               for e in range(len(idx_ca))]) for r in radius_grid}
    ftu_per_r = {r: fused_per_r[r][test_flat_idx] for r in radius_grid}

    # 2D joint CV at alpha_cv only
    sub_ca = stratified_sub(y[idx_ca], max_cal, seed*100+43)
    n_sub = len(sub_ca)
    coords_sub = np.stack([cal_flat_idx[sub_ca] // W, cal_flat_idx[sub_ca] % W], 1).astype(float)
    y_sub = y[idx_ca][sub_ca]
    cal_sub_flat = cal_flat_idx[sub_ca]
    fcu_sub_per_r = {r: fcu_per_r[r][sub_ca] for r in radius_grid}

    bws_valid = [bw for bw in bw_grid if bw < Seff * 0.8]
    if not bws_valid: bws_valid = [max(2, Seff // 20)]

    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=seed)
    cv_is_sacp  = {r: [] for r in radius_grid}
    cv_is_geocp = {(r, bw): [] for r in radius_grid for bw in bws_valid}
    for f_tr_idx, f_val_idx in kf.split(np.arange(n_sub)):
        y_cv_val = y_sub[f_val_idx]
        d_cv = cdist(coords_sub[f_val_idx], coords_sub[f_tr_idx])
        for r in radius_grid:
            fcu_tr = fcu_sub_per_r[r][f_tr_idx]
            fcu_val_all = fused_per_r[r][cal_sub_flat[f_val_idx]]
            # B: SACP global τ
            q_fold = conformal_quantile(fcu_tr, alpha_cv)
            cv_is_sacp[r].append(
                set_interval_score_vec(fcu_val_all, np.full(len(f_val_idx), q_fold), y_cv_val, alpha_cv))
            # C: GeoCP at (r, bw)
            order_tr = np.argsort(fcu_tr); sorted_tr = fcu_tr[order_tr]
            for bw in bws_valid:
                q_val = vwq(sorted_tr, d_cv, order_tr, bw, alpha_cv)
                cv_is_geocp[(r, bw)].append(
                    set_interval_score_vec(fcu_val_all, q_val, y_cv_val, alpha_cv))
    cv_sacp_mean  = {r: float(np.mean(v))  for r, v in cv_is_sacp.items()}
    cv_geocp_mean = {k: float(np.mean(v)) for k, v in cv_is_geocp.items()}
    best_r_sacp = int(min(cv_sacp_mean, key=cv_sacp_mean.get))
    best_rb     = min(cv_geocp_mean, key=cv_geocp_mean.get)
    best_r_geo, best_bw_geo = int(best_rb[0]), int(best_rb[1])

    # Test GeoCP weighted quantile at each α (distance matrix reused)
    fcu_C = fcu_per_r[best_r_geo][sub_ca]
    ftu_C = ftu_per_r[best_r_geo]
    order_C = np.argsort(fcu_C); sorted_C = fcu_C[order_C]
    coords_te = np.stack([test_flat_idx // W, test_flat_idx % W], 1).astype(float)
    n_te = len(test_flat_idx); batch_test = min(2000, max(200, n_te))
    q_C_per_alpha = {a: np.empty(n_te) for a in alpha_grid}
    for b in range(0, n_te, batch_test):
        be = min(b + batch_test, n_te)
        d = cdist(coords_te[b:be], coords_sub)
        for a in alpha_grid:
            q_C_per_alpha[a][b:be] = vwq(sorted_C, d, order_C, best_bw_geo, a)

    # α loop
    per_alpha = {}
    y_te_arr = y[idx_te]
    for a in alpha_grid:
        q_D_s = conformal_quantile(cal_true, a)
        q_A_s = conformal_quantile(fcu_per_r[1], a)
        q_B_s = conformal_quantile(fcu_per_r[best_r_sacp], a)
        blocks = {
            'D': eval_method(test_all,           q_D_s, y_te_arr, a),
            'A': eval_method(ftu_per_r[1],       q_A_s, y_te_arr, a),
            'B': eval_method(ftu_per_r[best_r_sacp], q_B_s, y_te_arr, a),
            'C': eval_method(ftu_C,              q_C_per_alpha[a], y_te_arr, a),
        }
        row = {}
        for lab, bk in blocks.items():
            row[lab] = {'cov': bk['cov'], 'size': bk['size'], 'is': bk['is_'],
                        'sscv_pct': bk['sscv_pct'], 'sscv_per_bucket': bk['sscv_per_bucket'],
                        'in_set': bk['in_set'].astype(bool),  # (n_te, K)
                        'sizes':   bk['sizes'].astype(np.int32),
                        'covered': bk['covered'].astype(bool)}
        row['q_C'] = q_C_per_alpha[a].astype(np.float32)
        per_alpha[f'{a:.2f}'] = row

    result = dict(
        tile=tile_key, size_px=int(Seff), size_km=Seff*0.01,
        H=int(H), W=int(W), n_classes=int(K),
        n_cal=int(len(idx_ca)), n_cal_sub=int(n_sub), n_test=int(len(idx_te)),
        alpha_grid=list(alpha_grid), alpha_cv=float(alpha_cv),
        accuracy=acc,
        best_r_sacp=best_r_sacp, best_r_geocp=best_r_geo, best_bw_geocp=best_bw_geo,
        per_alpha=per_alpha,
        cv_is_sacp=cv_sacp_mean,
        cv_is_geocp={f'{r}_{bw}': v for (r,bw), v in cv_geocp_mean.items()},
        probs_ca=probs_ca.astype(np.float32), probs_te=probs_te.astype(np.float32),
        cal_all_aps=cal_all.astype(np.float32), test_all_aps=test_all.astype(np.float32),
        cal_true_aps=cal_true.astype(np.float32),
        fcu_per_r={int(r): v.astype(np.float32) for r, v in fcu_per_r.items()},
        ftu_per_r={int(r): v.astype(np.float32) for r, v in ftu_per_r.items()},
        cal_flat_idx=cal_flat_idx.astype(np.int64),
        test_flat_idx=test_flat_idx.astype(np.int64),
        y_ca=y[idx_ca].astype(np.int64), y_te=y_te_arr.astype(np.int64),
        pred_te=pred_te.astype(np.int64),
        label=label.astype(np.int32),
    )
    tmp = ckpt_path + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, ckpt_path)
    return result

print('run_tile_size defined')


run_tile_size defined


## 6. Run all 10 tiles × 3 sizes (resumable, mirrors to Drive)

In [6]:
def _mirror(tile, size_px):
    src = f'{CKPT_DIR}/{tile}_s{size_px}.pkl'
    dst = f'{DRIVE_CKPT_DIR}/{tile}_s{size_px}.pkl'
    try:
        os.makedirs(DRIVE_CKPT_DIR, exist_ok=True); shutil.copy2(src, dst); return True
    except OSError: return False

total = len(TILES) * len(SIZES); done = 0; t_all = time.time()
log_file = f'{RESULTS_DIR}/run_log.txt'
for tile in TILES:
    print(f'\n{"="*80}\n{tile}  ({TILES[tile]})\n{"="*80}')
    for S in SIZES:
        ckpt = f'{CKPT_DIR}/{tile}_s{S}.pkl'
        if os.path.exists(ckpt):
            try:
                with open(ckpt, 'rb') as f: r = pickle.load(f)
            except OSError as e:
                print(f'  {S:>4d}px cached-read FAILED ({e}) — will recompute.')
            else:
                done += 1
                if r.get('degenerate'):
                    print(f'  {S:>4d}px [cached, degenerate K={r["n_classes"]}]  [{done}/{total}]')
                else:
                    pa = r['per_alpha']
                    print(f'  {S:>4d}px [cached]  acc={r["accuracy"]:.3f}  '
                          f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                          f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                          f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  [{done}/{total}]')
                continue
        t0 = time.time()
        try:
            r = run_tile_size(tile, S)
            done += 1
            mirrored = _mirror(tile, S)
            if r.get('degenerate'):
                msg = f'  {S:>4d}px DEGENERATE (K={r["n_classes"]})  [{time.time()-t0:.0f}s]  [{done}/{total}]'
            else:
                pa = r['per_alpha']
                msg = (f'  {S:>4d}px acc={r["accuracy"]:.3f}  '
                       f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                       f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                       f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  '
                       f'[{time.time()-t0:.0f}s]  [{done}/{total}]  '
                       f'{"mirrored" if mirrored else "(mirror skipped)"}')
            print(msg)
            try:
                with open(log_file, 'a') as f: f.write(f'{tile} {msg}\n')
            except OSError: pass
        except Exception as e:
            import traceback; print(f'  {S:>4d}px FAILED: {e}'); traceback.print_exc()
            try:
                with open(log_file, 'a') as f: f.write(f'{tile} s={S} FAILED: {e}\n')
            except OSError: pass

print(f'\n{"="*80}\nALL DONE: {done}/{total} configs  ({(time.time()-t_all)/60:.1f} min)')



polk_iowa  (Polk County, Iowa (row-crop))
   100px acc=0.844  α=0.10 IS: D=3.265 A=2.939 B=2.990 C=2.962  (r*=2,bw*=50)  [11s]  [1/30]  mirrored
   200px acc=0.822  α=0.10 IS: D=3.438 A=3.265 B=3.256 C=3.288  (r*=5,bw*=10)  [72s]  [2/30]  mirrored
   500px acc=0.815  α=0.10 IS: D=3.542 A=3.409 B=3.337 C=3.355  (r*=2,bw*=50)  [600s]  [3/30]  mirrored

lancaster_pa  (Lancaster, Pennsylvania (mixed ag))
   100px acc=0.873  α=0.10 IS: D=3.241 A=3.361 B=3.214 C=3.379  (r*=10,bw*=20)  [6s]  [4/30]  mirrored
   200px acc=0.840  α=0.10 IS: D=3.435 A=3.445 B=3.287 C=3.289  (r*=5,bw*=10)  [71s]  [5/30]  mirrored
   500px acc=0.798  α=0.10 IS: D=3.512 A=3.435 B=3.321 C=3.372  (r*=2,bw*=20)  [594s]  [6/30]  mirrored

hartford_ct  (Hartford, Connecticut (urban-forest))
   100px acc=0.762  α=0.10 IS: D=3.809 A=3.665 B=3.673 C=3.663  (r*=2,bw*=50)  [6s]  [7/30]  mirrored
   200px acc=0.777  α=0.10 IS: D=3.723 A=3.423 B=3.446 C=3.414  (r*=2,bw*=50)  [69s]  [8/30]  mirrored
   500px acc=0.782  α=0.10 

## 7. Aggregate → CSV + paired tests per α

In [7]:
rows = []
for tile in TILES:
    for S in SIZES:
        p = f'{CKPT_DIR}/{tile}_s{S}.pkl'
        if not os.path.exists(p): continue
        with open(p, 'rb') as f: r = pickle.load(f)
        if r.get('degenerate'): continue
        for a_str, block in r['per_alpha'].items():
            a = float(a_str)
            for lab in ('D','A','B','C'):
                b = block[lab]
                rows.append({'tile': tile, 'size_px': r['size_px'], 'size_km': r['size_km'],
                             'alpha': a, 'method': lab,
                             'is': b['is'], 'cov': b['cov'], 'size': b['size'], 'sscv_pct': b['sscv_pct'],
                             'accuracy': r['accuracy'],
                             'r_B': r['best_r_sacp'], 'r_C': r['best_r_geocp'], 'bw_C': r['best_bw_geocp']})
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/per_config_alpha.csv', index=False)
print('wrote', f'{RESULTS_DIR}/per_config_alpha.csv  ({len(df)} rows)')

pooled = (df.groupby(['alpha','method'])
            .agg(is_mean=('is','mean'), is_std=('is','std'),
                 cov_mean=('cov','mean'), size_mean=('size','mean'),
                 sscv_mean=('sscv_pct','mean'), sscv_std=('sscv_pct','std'),
                 n=('tile','count')).reset_index())
pooled.to_csv(f'{RESULTS_DIR}/alpha_sweep_pooled.csv', index=False)
print('\n=== Pooled α-sweep ===')
print(pooled.round(3).to_string(index=False))

print('\n=== Paired tests (C vs A, C vs D) per α ===')
pt_rows = []
for a in sorted(df.alpha.unique()):
    for ref in ('A','D'):
        sub_ref = df[(df.alpha==a)&(df.method==ref)].sort_values(['tile','size_px']).reset_index(drop=True)
        sub_C   = df[(df.alpha==a)&(df.method=='C')].sort_values(['tile','size_px']).reset_index(drop=True)
        impr = (sub_ref['is'].values - sub_C['is'].values) / sub_ref['is'].values * 100
        t, p = sstats.ttest_1samp(impr, 0.0)
        pt_rows.append({'alpha': a, 'comparison': f'C vs {ref}', 'mean_pct': impr.mean(),
                        'sem': impr.std(ddof=1)/np.sqrt(len(impr)), 't': t, 'p': p, 'n': len(impr)})
        print(f'  α={a:.2f}  C vs {ref}: {impr.mean():+5.2f}% ± {impr.std(ddof=1)/np.sqrt(len(impr)):.2f}  t={t:.2f}  p={p:.2e}  (n={len(impr)})')
pd.DataFrame(pt_rows).to_csv(f'{RESULTS_DIR}/alpha_sweep_paired_tests.csv', index=False)


wrote /content/s2_alpha_sweep/results/per_config_alpha.csv  (348 rows)

=== Pooled α-sweep ===
 alpha method  is_mean  is_std  cov_mean  size_mean  sscv_mean  sscv_std  n
  0.05      A    3.466   0.349     0.950      1.467      5.119     0.662 29
  0.05      B    3.373   0.375     0.951      1.416      4.829     1.938 29
  0.05      C    3.377   0.360     0.951      1.433      4.920     1.006 29
  0.05      D    3.585   0.370     0.949      1.535      5.017     1.035 29
  0.10      A    3.268   0.236     0.900      1.267      9.996     0.021 29
  0.10      B    3.208   0.228     0.899      1.198      9.137     1.531 29
  0.10      C    3.199   0.237     0.901      1.214      9.201     1.235 29
  0.10      D    3.337   0.283     0.898      1.306      9.844     0.414 29
  0.15      A    3.171   0.153     0.848      1.141     15.115     0.619 29
  0.15      B    3.090   0.154     0.849      1.071     13.859     3.726 29
  0.15      C    3.089   0.157     0.850      1.084     13.737     1.

## 8. Package & mirror final zip to Drive

In [8]:
zip_local = '/content/s2_alpha_sweep_results.zip'
shutil.make_archive('/content/s2_alpha_sweep_results', 'zip', LOCAL_DIR)
print('zipped to', zip_local)
try:
    os.makedirs(DRIVE_MIRROR, exist_ok=True)
    shutil.copy2(zip_local, f'{DRIVE_MIRROR}/s2_alpha_sweep_results.zip')
    print('copied to Drive:', f'{DRIVE_MIRROR}/s2_alpha_sweep_results.zip')
except OSError as e:
    print(f'Drive copy failed ({e}); manual download: files.download("/content/s2_alpha_sweep_results.zip")')


zipped to /content/s2_alpha_sweep_results.zip
copied to Drive: /content/drive/MyDrive/s2_alpha_sweep/s2_alpha_sweep_results.zip
